### MuSiQue Corpus 만들기

In [ ]:
import pandas as pd
import ast
import json
from tqdm import tqdm

# ==========================================
# 1. 전역 변수 초기화 (Global Deduplication용)
# ==========================================
# (Title, Text) 튜플을 Key로, 부여된 ID를 Value로 저장
passage_to_id_map = {} 
# 최종 저장할 고유 Passage 리스트
corpus_list = [] 

def process_and_map_passages(paragraphs_str):
    """
    각 행의 paragraphs 문자열을 파싱하여:
    1. 전역 Corpus에 없는 passage면 등록하고 ID 부여
    2. is_supporting=True인 passage의 ID를 리스트로 반환
    """
    try:
        # 문자열을 리스트로 변환 ('[{...}]' -> [{...}])
        passages = ast.literal_eval(paragraphs_str)
    except (ValueError, SyntaxError):
        return []

    gt_indices = []

    for p in passages:
        # 고유성 판단 기준: 제목(Title)과 본문(Text)의 조합
        # 띄어쓰기 등 미세한 차이를 무시하려면 .strip() 추가 가능
        unique_key = (p['title'], p['paragraph_text'])

        # 1. 전역 Corpus에 등록되어 있는지 확인
        if unique_key not in passage_to_id_map:
            # 없으면 새로 등록
            new_id = len(corpus_list) # 현재 리스트 길이가 곧 새로운 ID (0부터 시작)
            passage_to_id_map[unique_key] = new_id
            
            # Corpus에 추가 (요청하신 포맷: passage_index, passage_text)
            corpus_list.append({
                "passage_index": new_id,
                # Retrieval용 Text는 보통 "Title: Content" 형태로 합칩니다.
                "passage_text": f"{p['title']}: {p['paragraph_text']}" 
            })
        
        # 2. 현재 Passage의 Global ID 가져오기
        current_pid = passage_to_id_map[unique_key]

        # 3. 정답(Supporting Fact)인지 확인하여 인덱스 저장
        if p.get('is_supporting', False):
            gt_indices.append(current_pid)

    return gt_indices

# ==========================================
# 2. DataFrame 적용
# ==========================================
# 진행 상황을 보기 위해 tqdm 사용 (데이터가 많을 경우 유용)
tqdm.pandas(desc="Processing Paragraphs")

df = pd.read_csv("/workspace/daeyong/benchmarks/musique_dev.csv")

# 실제 적용: 각 행마다 process_and_map_passages 함수 실행
# 결과는 df['gt_passage_index'] 컬럼에 [3, 107] 형태로 저장됨
df['gt_passage_index'] = df['paragraphs'].progress_apply(process_and_map_passages)

# ==========================================
# 3. 결과 확인 및 저장
# ==========================================
print(f"✅ Processing Complete.")
print(f"📊 Total Unique Passages created: {len(corpus_list)}")

# DataFrame 결과 확인 (상위 5개)
print("\nDataFrame Head with GT Indices:")
print(df[['gt_passage_index']].head())

# df.to_csv("/workspace/daeyong/benchmarks/musique_dev.csv", index=False)

# Corpus를 JSON 파일로 저장
corpus_output_path = "/workspace/daeyong/benchmarks/musique_corpus.json"
with open(corpus_output_path, "w", encoding="utf-8") as f:
    json.dump(corpus_list, f, indent=2, ensure_ascii=False)

print(f"\n💾 Unique Corpus saved to: {corpus_output_path}")

### Hotpotqa corpus 만들기

In [ ]:
import pandas as pd
import ast
import json
from tqdm import tqdm

# ==========================================
# 1. 전역 변수 초기화
# ==========================================
# (Title, Text) 튜플을 Key로, 부여된 Global ID를 Value로 저장
passage_to_id_map = {} 
unique_corpus = []

def process_hotpotqa_row(row):
    """
    각 행을 처리하여:
    1. Context 내 문서들을 Global Corpus에 등록 (중복 제거)
    2. Supporting Facts(GT)에 해당하는 문서들의 Global ID 리스트 반환
    """
    try:
        # 문자열 -> 딕셔너리 변환
        context = ast.literal_eval(row['context'])
        sp_facts = ast.literal_eval(row['supporting_facts'])
    except (ValueError, SyntaxError):
        return []

    # 1. 정답 문서의 Title 집합 만들기 (Lookup 속도 향상)
    # supporting_facts 포맷: {'title': ['A', 'B'], 'sent_id': [0, 0]}
    gt_titles = set(sp_facts.get('title', []))
    
    current_row_gt_indices = set()

    # context의 title과 sentences는 리스트 순서가 서로 대응됨
    titles = context.get('title', [])
    sentences_list = context.get('sentences', [])

    # 2. Context 내의 각 문서를 순회
    for title, sents in zip(titles, sentences_list):
        # 문장 리스트를 하나의 문단으로 합치기 (요청사항)
        paragraph_text = "".join(sents)
        
        # 고유성 판단 키: (제목, 본문)
        unique_key = (title, paragraph_text)

        # --- Global Corpus 등록 ---
        if unique_key not in passage_to_id_map:
            new_id = len(unique_corpus)
            passage_to_id_map[unique_key] = new_id
            
            # Corpus에 추가 ("Title: Text" 포맷 권장)
            unique_corpus.append({
                "passage_index": new_id,
                "passage_text": f"{title}: {paragraph_text}"
            })
        
        # 현재 문서의 Global ID 가져오기
        global_id = passage_to_id_map[unique_key]

        # --- GT 확인 ---
        # 현재 문서의 제목이 정답(supporting_facts) 제목 목록에 있다면 GT로 추가
        if title in gt_titles:
            current_row_gt_indices.add(global_id)

    return list(current_row_gt_indices)

# ==========================================
# 2. DataFrame 적용
# ==========================================
tqdm.pandas(desc="Indexing & Mapping HotpotQA")

# df['gt_passage_index'] 컬럼 생성
df['gt_passage_index'] = df.progress_apply(process_hotpotqa_row, axis=1)

# ==========================================
# 3. 결과 확인 및 저장
# ==========================================
print(f"✅ Processing Complete.")
print(f"📊 Total Unique Passages created: {len(unique_corpus)}")

# 결과 확인
print("\nDataFrame Head with GT Indices:")
print(df[['gt_passage_index']].head())

# df.to_csv("/workspace/daeyong/benchmarks/hotpotqa_dev.csv", index=False)

# Corpus JSON 저장
corpus_output_path = "/workspace/daeyong/benchmarks/hotpotqa_corpus.json"
with open(corpus_output_path, "w", encoding="utf-8") as f:
    json.dump(unique_corpus, f, indent=2, ensure_ascii=False)

print(f"\n💾 Unique Corpus saved to: {corpus_output_path}")

### 2Wiki corpus 만들기

In [ ]:
df = pd.read_csv("/workspace/daeyong/benchmarks/2wiki_dev.csv")
df

In [ ]:
import pandas as pd
import ast
import json
from tqdm import tqdm

# ==========================================
# 1. 전역 변수 초기화
# ==========================================
# (Title, Text) 튜플을 Key로, 부여된 Global ID를 Value로 저장
passage_to_id_map = {} 
unique_corpus = []

def process_2wiki_row(row):
    """
    2WikiMultihopQA 스타일의 데이터 처리:
    1. Context 리스트를 순회하며 Global Corpus에 등록 (중복 제거)
    2. Supporting Facts(Title 기준)에 해당하는 문서의 Global ID 매핑
    """
    try:
        # 문자열 -> 파이썬 객체 변환
        context = ast.literal_eval(row['context'])
        sp_facts = ast.literal_eval(row['supporting_facts'])
    except (ValueError, SyntaxError):
        return []

    # 1. 정답 문서의 Title 집합 만들기
    # supporting_facts 구조: [['Title', sent_id], ['Title2', sent_id]]
    # 우리가 필요한 건 Title 뿐입니다.
    gt_titles = set(item[0] for item in sp_facts)
    
    current_row_gt_indices = set()

    # 2. Context 순회
    # Context 구조: [['Title', ['Sent1', 'Sent2']], ['Title2', ...]]
    for doc in context:
        title = doc[0]
        sentences = doc[1]
        
        # 문장 리스트를 하나의 문단으로 합치기 (요청: 공백으로 join)
        paragraph_text = " ".join(sentences)
        
        # 고유성 판단 키: (제목, 본문)
        unique_key = (title, paragraph_text)

        # --- Global Corpus 등록 ---
        if unique_key not in passage_to_id_map:
            new_id = len(unique_corpus)
            passage_to_id_map[unique_key] = new_id
            
            # Corpus에 추가 ("Title: Text" 포맷)
            unique_corpus.append({
                "passage_index": new_id,
                "passage_text": f"{title}: {paragraph_text}"
            })
        
        # 현재 문서의 Global ID 가져오기
        global_id = passage_to_id_map[unique_key]

        # --- GT 확인 ---
        # 현재 문서의 제목이 정답(gt_titles)에 포함되어 있으면 GT Index로 추가
        if title in gt_titles:
            current_row_gt_indices.add(global_id)

    return list(current_row_gt_indices)

# ==========================================
# 2. DataFrame 적용
# ==========================================
tqdm.pandas(desc="Indexing & Mapping 2Wiki")

# df['gt_passage_index'] 컬럼 생성
df['gt_passage_index'] = df.progress_apply(process_2wiki_row, axis=1)

# ==========================================
# 3. 결과 확인 및 저장
# ==========================================
print(f"✅ Processing Complete.")
print(f"📊 Total Unique Passages created: {len(unique_corpus)}")

# 결과 확인
print("\nDataFrame Head with GT Indices:")
print(df[['gt_passage_index']].head())

# df.to_csv("/workspace/daeyong/benchmarks/2wiki_dev.csv", index=False)

# Corpus JSON 저장
corpus_output_path = "/workspace/daeyong/benchmarks/2wiki_corpus.json"
with open(corpus_output_path, "w", encoding="utf-8") as f:
    json.dump(unique_corpus, f, indent=2, ensure_ascii=False)

print(f"\n💾 Unique Corpus saved to: {corpus_output_path}")